In [ ]:
import os
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import torchaudio.transforms as T
import torchvision.models as models
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter

In [ ]:
def get_data_dir():
    data_dir = os.environ.get("VIMD_DATA_DIR")
    if not data_dir:
        raise ValueError("Set VIMD_DATA_DIR to the dataset root directory.")
    return data_dir


DATA_DIR = get_data_dir()
TRAIN_CSV_PATH = os.path.join(DATA_DIR, "train", "vimd_metadata.csv")
VALID_CSV_PATH = os.path.join(DATA_DIR, "validation", "vimd_metadata.csv")
TEST_CSV_PATH = os.path.join(DATA_DIR, "test", "vimd_metadata.csv")

MAX_EPOCHS = 500
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
PATIENCE = 15
TARGET_SAMPLE_RATE = 16000
DURATION = 3
NUM_SAMPLES = TARGET_SAMPLE_RATE * DURATION
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def print_tree_folders_only(path, prefix="", max_folders=10):
    if not os.path.isdir(path):
        return

    print(prefix + os.path.basename(path) + "/")
    prefix_child = prefix + "│   "

    subdirs = [
        d for d in sorted(os.listdir(path))
        if os.path.isdir(os.path.join(path, d))
    ]

    for d in subdirs[:max_folders]:
        print_tree_folders_only(
            os.path.join(path, d),
            prefix_child,
            max_folders
        )

    if len(subdirs) > max_folders:
        print(prefix_child + f"... ({len(subdirs) - max_folders} folders omitted)")


# Optional sanity check for dataset structure.
print_tree_folders_only(DATA_DIR, max_folders=10)

In [ ]:
df = pd.read_csv(TRAIN_CSV_PATH)
df.head()

In [ ]:
def normalize_label(value):
    if not isinstance(value, str):
        return value
    text = value.strip().lower()
    text = "".join(
        ch for ch in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(ch)
    )
    return text


def encode_gender_label(gender_value):
    if pd.isna(gender_value):
        raise ValueError("Missing value in gender column")

    if isinstance(gender_value, str):
        gender_value = normalize_label(gender_value)
        label_map = {
            "female": 0,
            "f": 0,
            "woman": 0,
            "nu": 0,
            "0": 0,
            "male": 1,
            "m": 1,
            "man": 1,
            "nam": 1,
            "1": 1,
        }
        if gender_value not in label_map:
            raise ValueError(f"Unknown gender value: {gender_value}")
        return label_map[gender_value]

    return int(gender_value)


def resolve_audio_path(row, split_name, data_dir):
    audio_dir = os.path.join(data_dir, split_name, "audio_vimd")
    candidates = []
    if "filename" in row and pd.notna(row["filename"]):
        candidates.append(os.path.join(audio_dir, str(row["filename"])))

    if "audio_saved_path" in row and pd.notna(row["audio_saved_path"]):
        saved_path = str(row["audio_saved_path"])
        if os.path.isabs(saved_path):
            candidates.append(saved_path)
        else:
            candidates.append(os.path.join(data_dir, saved_path))
        candidates.append(os.path.join(audio_dir, os.path.basename(saved_path)))

    if "id" in row and pd.notna(row["id"]):
        try:
            candidates.append(
                os.path.join(audio_dir, f"{int(row['id']):06d}.wav")
            )
        except (TypeError, ValueError):
            pass

    for path in candidates:
        if path and os.path.exists(path):
            return path

    return candidates[0] if candidates else None


def load_vimd_split(split_name, data_dir):
    csv_path = os.path.join(data_dir, split_name, "vimd_metadata.csv")
    df_split = pd.read_csv(csv_path)

    required_cols = {"gender", "id", "filename", "split"}
    missing_cols = required_cols - set(df_split.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns in {csv_path}: {missing_cols}")

    df_split = df_split.copy()
    df_split["split"] = split_name
    df_split["label"] = df_split["gender"].apply(encode_gender_label).astype(np.int64)
    df_split["filepath"] = df_split.apply(
        lambda row: resolve_audio_path(row, split_name, data_dir),
        axis=1
    )

    return df_split[["filepath", "split", "label", "gender", "id", "filename"]]


df_all = pd.concat(
    [
        load_vimd_split("train", DATA_DIR),
        load_vimd_split("validation", DATA_DIR),
        load_vimd_split("test", DATA_DIR),
    ],
    ignore_index=True,
)

print(df_all.groupby(["split", "label"]).size())
df_all.head()

In [ ]:
class GenderAudioDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = self.df.loc[idx, "filepath"]
        label = self.df.loc[idx, "label"]
        waveform, sr = torchaudio.load(path)

        if sr != TARGET_SAMPLE_RATE:
            resampler = T.Resample(sr, TARGET_SAMPLE_RATE)
            waveform = resampler(waveform)

        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        if waveform.shape[1] > NUM_SAMPLES:
            waveform = waveform[:, :NUM_SAMPLES]
        elif waveform.shape[1] < NUM_SAMPLES:
            pad_amount = NUM_SAMPLES - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad_amount))

        if self.transform:
            mel_spec = self.transform(waveform)
        else:
            mel_spec = waveform

        return mel_spec, label


mel_transform = T.MelSpectrogram(
    sample_rate=TARGET_SAMPLE_RATE,
    n_fft=1024,
    hop_length=512,
    n_mels=64,
)

In [ ]:
train_df = df_all[df_all["split"] == "train"]
valid_df = df_all[df_all["split"] == "validation"]
test_df = df_all[df_all["split"] == "test"]

train_dataset = GenderAudioDataset(train_df, transform=mel_transform)
valid_dataset = GenderAudioDataset(valid_df, transform=mel_transform)
test_dataset = GenderAudioDataset(test_df, transform=mel_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
)

In [ ]:
class CNNBiLSTM(nn.Module):
    def __init__(self, num_classes=2, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.lstm_input_size = 128 * 8
        self.bilstm = nn.LSTM(
            input_size=self.lstm_input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2, num_classes),
        )

    def forward(self, x):
        x = self.cnn(x)
        x = x.permute(0, 3, 1, 2)
        x = x.reshape(x.size(0), x.size(1), -1)
        out, _ = self.bilstm(x)
        out = out[:, -1, :]
        out = self.classifier(out)
        return out


def get_model(model_type):
    if model_type == 1:
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        model.fc = nn.Linear(model.fc.in_features, 2)
    elif model_type == 2:
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        model.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
    elif model_type == 3:
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        model.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
    elif model_type == 4:
        model = CNNBiLSTM(num_classes=2)
    else:
        raise ValueError(
            "model_type must be 1=ResNet18, 2=MobileNetV2, 3=EfficientNetB0, 4=CNN-BiLSTM"
        )
    return model

In [ ]:
class EarlyStopping:
    def __init__(self, patience=15, delta=0, path="best_model.pth"):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta
        self.path = path

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

In [ ]:
def train_model(model, train_loader, valid_loader, model_name):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    writer = SummaryWriter(f"runs/{model_name}")
    early_stopping = EarlyStopping(patience=PATIENCE, path=f"{model_name}_best.pth")

    for epoch in range(MAX_EPOCHS):
        model.train()
        train_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * inputs.size(0)

        train_loss = train_loss / len(train_loader.dataset)

        model.eval()
        valid_loss = 0.0
        val_preds = []
        val_targets = []
        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                valid_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(labels.cpu().numpy())

        valid_loss = valid_loss / len(valid_loader.dataset)
        val_acc = accuracy_score(val_targets, val_preds)

        writer.add_scalar("Loss/train", train_loss, epoch)
        writer.add_scalar("Loss/valid", valid_loss, epoch)
        writer.add_scalar("Accuracy/valid", val_acc, epoch)

        early_stopping(valid_loss, model)
        if early_stopping.early_stop:
            break

    writer.close()
    model.load_state_dict(torch.load(f"{model_name}_best.pth"))
    return model

In [ ]:
def evaluate_and_visualize(model, test_loader, model_name):
    model.eval()
    test_preds = []
    test_targets = []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            test_preds.extend(preds.cpu().numpy())
            test_targets.extend(labels.cpu().numpy())

    acc = accuracy_score(test_targets, test_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        test_targets,
        test_preds,
        average="binary",
    )

    print(f"Metrics for {model_name}:")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

    cm = confusion_matrix(test_targets, test_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["female", "male"],
        yticklabels=["female", "male"],
    )
    plt.title(f"Confusion Matrix - {model_name}")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.show()

In [ ]:
def run(model_type):
    model_names = {
        1: "ResNet18",
        2: "MobileNetV2",
        3: "EfficientNetB0",
        4: "CNN-BiLSTM",
    }
    model_name = model_names.get(model_type, "Unknown_Model")

    model = get_model(model_type)
    model = model.to(DEVICE)

    best_model = train_model(model, train_loader, valid_loader, model_name)
    evaluate_and_visualize(best_model, test_loader, model_name)

In [ ]:
run(1)

In [ ]:
run(2)

In [ ]:
run(3)

In [ ]:
run(4)